# SOMD 2026 — Mention Masking Noise Injection
Replaces all mention strings with a `[MASK]` token, completely removing
surface-form information. This is a diagnostic perturbation — not a realistic
detection error — designed to isolate how much document context alone
contributes to coreference decisions, independently of lexical content.

## 0. Imports & Setup

In [ ]:
import json
from copy import deepcopy
from tqdm import tqdm
import os
import random

MASK_TOKEN = "[MASK]"

## 1. Load Data

In [ ]:
# ── Paths — adjust as needed ──────────────────────────────────────────────────
SUBTASK = "subtask 1"  # change to "subtask 2" for Subtasks 2 & 3

file_to_load = f"../data/SOMD 2026/{SUBTASK}/train_data.jsonl"
with open(file_to_load, 'r') as f:
    train_data = [json.loads(l) for l in f]

with open(f"../data/SOMD 2026/{SUBTASK}/train_labels.json", "r") as f:
    train_labels = json.load(f)

print(f"Loaded {len(train_data):,} mentions")
print(f"Loaded {len(train_labels):,} clusters")
print(f"\nExample mention:")
print(json.dumps(train_data[0], indent=2))

## 2. Noise Function — Mention Masking

For each selected mention, the mention string is replaced with `[MASK]`
and the sentence is updated accordingly. Span boundaries are updated to
reflect the fixed length of the mask token.

At 100% noise rate, **all** mention strings are masked, leaving only
surrounding document context as a coreference signal. This is the
strongest diagnostic condition: any system that maintains non-trivial
performance here is genuinely exploiting context rather than surface form.

In [ ]:
def inject_mention_masking_noise(train_data, noise_rate, mask_token=MASK_TOKEN):
    """
    Replace mention strings with a [MASK] token.

    For each selected mention:
    - Replace the mention string with mask_token
    - Update the sentence text and span boundaries accordingly

    Args:
        train_data  : list of mention dicts
        noise_rate  : fraction of mentions to mask (0.0 to 1.0)
        mask_token  : replacement token (default: '[MASK]')

    Returns:
        Perturbed copy of train_data
    """
    new_train_data = deepcopy(train_data)

    n_to_modify = int(len(new_train_data) * noise_rate)
    indices_to_modify = set(random.sample(range(len(new_train_data)), n_to_modify))

    stats = {'masked': 0}

    for idx, mention in enumerate(tqdm(new_train_data, desc="Injecting mention masking noise")):
        if idx not in indices_to_modify:
            continue

        sentence = mention['sentence']
        start    = mention['start']
        end      = mention['end']

        # Replace mention span in sentence with mask token
        new_sentence = sentence[:start] + mask_token + sentence[end:]
        new_end      = start + len(mask_token)

        mention['mention']  = mask_token
        mention['sentence'] = new_sentence
        mention['end']      = new_end
        # start remains unchanged

        stats['masked'] += 1

    print(f"\nMention masking noise injection stats:")
    total = stats['masked']
    print(f"  Masked : {total}/{len(new_train_data)} ({total/len(new_train_data):.1%})")

    return new_train_data

## 3. Quick Sanity Check

In [ ]:
# Run on a tiny sample to verify behaviour before full run
sample = train_data[:20]
noisy_sample = inject_mention_masking_noise(sample, noise_rate=1.0)

print("\n── Before / After comparison ──")
for orig, noisy in zip(sample[:5], noisy_sample[:5]):
    print(f"  ORIGINAL : '{orig['mention']}'")
    print(f"  MASKED   : '{noisy['mention']}'")
    print(f"  SENTENCE : ...{noisy['sentence'][max(0,noisy['start']-20):noisy['end']+20]}...")
    print()

# Verify span boundaries are correct
print("── Span boundary verification ──")
for orig, noisy in zip(sample[:5], noisy_sample[:5]):
    recovered = noisy['sentence'][noisy['start']:noisy['end']]
    assert recovered == MASK_TOKEN, f"Boundary error: got '{recovered}'"
print("All span boundaries verified correctly.")

## 4. Generate Noisy Datasets

In [ ]:
noise_rates = [0, 0.10, 0.25, 0.50, 0.75, 1.0]

output_dir = f"../../SOMD-2026/data/SOMD 2026/{SUBTASK}/noisy_data_masking"
os.makedirs(output_dir, exist_ok=True)

for noise_rate in noise_rates:
    print(f"\n{'='*50}")
    print(f"Generating noise rate: {noise_rate:.0%}")
    print(f"{'='*50}")

    noisy_data = inject_mention_masking_noise(train_data, noise_rate=noise_rate)

    subtask_tag = SUBTASK.replace(' ', '_')
    filename = os.path.join(
        output_dir,
        f"noisy_{noise_rate}_train_data_{subtask_tag}_masking.jsonl"
    )

    with open(filename, 'w') as f:
        for item in noisy_data:
            f.write(json.dumps(item) + '\n')

    print(f"Saved: {filename}")

print("\nAll done.")